In [ ]:
import os
os.environ["IFLOW_API_KEY"] = "sk-072e0c57421af7fcb87b31d8767e7057"

import sys
sys.path.append(r"D:\git-projects\vis-context-agent\agents")

from graph_agent.llms.llm import LLMTool
from graph_agent.llms.apis.iflow import IFlowApi

llm = IFlowApi(model="qwen3-max", max_tokens=16384)
llm_tool = LLMTool("qwen3-max", llm)
llm_tool.compile()

In [1]:
%reload_ext autoreload
%autoreload 2

from database import Database

DICT_DB_PATH = r"D:\git-projects\vis-context-agent\song-bureaucracy\data\database\song_bureaucracy_dictionary.db"
DICT_TABLE = "chapter8t10"
ENTRY_DB_PATH = r"D:\git-projects\vis-context-agent\song-bureaucracy\data\database\song_bureaucracy_entries_v0211.db"

db = Database(DICT_DB_PATH, DICT_TABLE, ENTRY_DB_PATH)

In [8]:
%reload_ext autoreload
%autoreload 2

from agent_state import AgentState
from prompt_input2facts import build_input2facts_prompt

# 1. 构建 AgentState 中的上下文/短期记忆
# 获取并存储《辞典》索引 标题-页码
# 维护待处理列表，辞典索引上下文
# 维护已获取的辞典条目
# 维护已获取更新的数据项信息
# 维护 CoT 记录

agent_state = AgentState(db=db)
dict_todo = list(agent_state.dict_index.keys())[0:1]
print(dict_todo)

for todo in dict_todo:
  agent_state.init_short_memory()
  title, page = todo.split("-")
  agent_state.dictionary_query(title, page)

  while not agent_state.finished:
    prompt = build_input2facts_prompt(
      dictionary_index_summary=agent_state.dict_index_text,
      current_dictionary_texts=agent_state.loaded_dict_entries.get_merged_text(),
      current_atomic_facts=agent_state.atomic_facts.get_merged_text(),
      history_summary=agent_state.cot.get_merged_text(),
    )
    print(prompt)
    break


['河北兵马大元帅府-482']

[系统角色定义]

你是一名精通宋代官制史的结构化数据标注专家，任务是：
根据提供的《宋代官制辞典》条目，总结其中涉及到的机构或官职的原子事实，用于后续下游任务更新宋代官制数据库。

你不直接修改数据库；你只负责：
1）判断当前信息是否形成闭环，没有引用其他《辞典》中的词条内容；若不闭环则对缺失的词条进行补充查询；
2）在充分获取信息后，按照辞典词条内容涉及到的各个机构或官职，总结相应的原子事实信息。

原子事实信息需要包含引用信息，涉及的机构或官职，时间点信息，以及属性或关系变化信息。
以半结构文本的形式组织，用于后续步骤构建宋代官制数据库，并不在这一轮的处理中涉及。

--------------------------------
[全局流程与基本准则]

1. 首先阅读「当前已知的辞典词条」，整体理解内容。
2. **严格判定是否需要补充查询**：
   - **必须补充查询的情况**：仅当当前词条文本中出现明确的**词条跳转指引**时（例如：“详见‘XX’条”、“内容同‘XX’”、“参见‘XX’词条”），且该被指向的词条目前不在上下文中，才允许调用 `search_dictionary`。
   - **禁止补充查询的情况**：若词条中仅仅是提到了其他机构或官职（例如：“下领XX案”、“受XX司统辖”、“与XX机构职权交叉”），只要这些关系在当前文本中描述清晰，**严禁**为了了解这些被提到机构的详细背景而发起查询。
3. **原子事实提取准则**：
   - 当不存在上述「显式条目跳转」或相关条目已获取时，立即开始提取原子事实。
   - 原子事实必须忠实于原文，描述机构/官职的属性变化、时间点及相互关系。

--------------------------------
[可用工具定义]

在 Action 阶段，你可以调用以下工具：

1）查询《辞典》原始文献：
- 名称：search_dictionary
- 语义：根据词条名称和页码，查询《宋代官制辞典》中对应的条目。
- 参数：
  - title: 词条名称（字符串，如 "都督府"）
  - page: 页码（字符串，如 "483"）
- 返回： 该工具会向上下文「当前已知的辞典词条」部分中追加新的辞典条目。

2）新增一条原子信息：
- 名称：add_a

In [ ]:
%reload_ext autoreload
%autoreload 2

from database import Database
from tools import (
  search_dictionary,
  dict_entry_to_str,
  get_entity,
  entity_to_str,
  create_entity,
  create_timepoint,
  update_timepoint_attr,
  create_timepoints_relationship,
  append_citation
)
from agent_state import AgentState

# 1. 创建 Database 实例，连接到辞典数据库和实体数据表
DICT_DB_PATH = r"D:\git-projects\vis-context-agent\song-bureaucracy\data\database\song_bureaucracy_dictionary.db"
DICT_TABLE = "chapter8t10"
ENTRY_DB_PATH = r"D:\git-projects\vis-context-agent\song-bureaucracy\data\database\song_bureaucracy_entries_v0211.db"

db = Database(DICT_DB_PATH, DICT_TABLE, ENTRY_DB_PATH)
# 初始化，清空数据表
db._recreate_tables()

# 2. 从辞典数据表中获取全部条目，构建待处理集合（title-page 作为键值）
cursor = db._dict_conn.cursor()
cursor.execute(f"SELECT id, title, page FROM {DICT_TABLE}")
rows = cursor.fetchall()
cursor.close()

entry_indexes = {}
for (id, title, page) in rows:
  entry_indexes[f"{title}-{page}"] = False

print(len(entry_indexes))

# 3. 构建 Agent 状态实例
agent_state = AgentState(db=db)

# 4. 保留函数调用接口：这里简单打印本轮处理的词条后结束

def run_agent_for_current_entry(state: AgentState) -> None:
  """外圈调用入口：根据当前状态执行 Agent 流程。"""
  print(state.dict_ctx.merged_text)
  return

# 使用循环：如果 pending_entries 非空，则取第一个作为本轮初始输入元素
for entry_key in entry_indexes:
  if entry_indexes[entry_key]:
    continue
  title, page = entry_key.split("-", 1)
  # 初始化当前轮的 Agent 状态
  agent_state.start_new_entry(title=title, page=page)
  entry = search_dictionary(db, title, page)
  agent_state.dict_ctx.add_page_entry(title, page, entry)
  # 调用 Agent 运行函数
  run_agent_for_current_entry(agent_state)
  break


833
# 河北兵马大元帅府
文本来源：宋代官制辞典/I.职官条目分类目录/第八编 军事统率机构与地方治安机构类/一、大元帅府、都督府门 482页
基本介绍：官司名。北宋靖康元年闰十一月，宋钦宗传檄，授命康王为河北兵马大元帅。十二月一日，赵构开大元帅府，以募兵勤王抗金，解救京师之围为名（《要录》卷1）。南宋建炎元年五月十日大元帅府解散（《宋会要·职官》37之2《元帅府》）。
简称: ①大元帅府、元帅府。《宋会要·职官》37之1：“高宗建炎元年五月二日，诏大元帅府限十日结局。”《要录》卷1靖康元年十一月己酉：“拜王（康王赵构)河北兵马大元帅。”十二月壬戌朔：“王开元帅府。”②帅府。《要录》卷1乙亥：“乃遣人伴送至帅府。”③霸府。《要录》卷1，靖康元年闰十一月己酉：“拜王河北兵马大元帅。”原注引《汪伯彦日历》：“然霸府肇启开，事出仓卒。盖靖康元年闰十一月，檄到日，康王可充兵马大元帅。”④天下兵马大元帅府。过称。《金佗粹编》卷4《行实编年》：“（靖康元年）冬，高宗皇帝以天下兵马大元帅开府河朔。”《要录》卷4，建炎元年四月癸亥：“（赵子崧）望大王遵故事，以天下兵马大元帅承制号召四方。”



In [4]:
import json
from typing import List, Dict, Any

def get_dictionary_index_summary(db: Database, dict_table: str) -> str:
  """
  从 database 获取辞典条目索引字符串。
  
  返回所有辞典条目的 title-page 组合列表，格式化为可读字符串，
  用于在提示词中提供辞典索引信息。
  
  Args:
    db: Database 实例
    dict_table: 辞典数据表名
    
  Returns:
    格式化的索引字符串，每行一个 "title-page" 条目
  """
  cursor = db._dict_conn.cursor()
  cursor.execute(f"SELECT id, title, page FROM {dict_table} ORDER BY id")
  rows = cursor.fetchall()
  cursor.close()
  
  entries = [f"{title}-{page}" for (id, title, page) in rows]
  summary = f"《辞典》中共有 {len(entries)} 条词条，索引如下：\n"
  summary += "\n".join(entries)
  return summary

print(get_dictionary_index_summary(db, DICT_TABLE)[:100])


《辞典》中共有 833 条词条，索引如下：
河北兵马大元帅府-482
河北兵马大元帅-482
河北兵马元帅-482
河北兵马副元帅-482
河北兵马大元帅府参议官-482
河北兵马大元帅府随军应副-4
